# Getting structured output

In [ ]:
# pip install langchain langchain-openai requests

In [14]:
import langchain
import requests
import importlib.metadata

print(f"LangChain version: {langchain.__version__}")
print(f"Requests version: {requests.__version__}")

try:
    l_openai_version = importlib.metadata.version("langchain-openai")
    print(f"LangChain-OpenAI version: {l_openai_version}")
except importlib.metadata.PackageNotFoundError:
    print("LangChain-OpenAI is not installed.")


LangChain version: 1.2.15
Requests version: 2.33.1
LangChain-OpenAI version: 1.2.1


In [ ]:
# Ideally put this in your enviroment

OPENAI_API_KEY = "key_here"

In [16]:
# Define Output Schemas

from pydantic import BaseModel

class EMIOutput(BaseModel):
    principal: float
    rate: float
    time: float
    emi: float

class SimpleInterestOutput(BaseModel):
    principal: float
    rate: float
    time: float
    interest: float


In [17]:
import math
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent

# LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=OPENAI_API_KEY,
    temperature=0
)

@tool
def calculate_emi(principal: float, rate: float, time: float) -> dict:
    """Calculate EMI for a loan."""
    
    print("...Running EMI calculator tool")

    R = rate / 12 / 100
    N = time * 12

    emi = (principal * R * (1 + R)**N) / ((1 + R)**N - 1)

    return EMIOutput(
        principal=principal,
        rate=rate,
        time=time,
        emi=round(emi, 2)
    ).model_dump()


@tool
def calculate_simple_interest(principal: float, rate: float, time: float) -> dict:
    """Calculate simple interest."""

    print("...Running simple interest tool")

    SI = (principal * rate * time) / 100

    return SimpleInterestOutput(
        principal=principal,
        rate=rate,
        time=time,
        interest=round(SI, 2)
    ).model_dump()


In [18]:
tools = [calculate_emi, calculate_simple_interest]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant with access to tools."
)

## Try
```
input1: "I took a loan of 2 lakh at 10% for 2 years, what's my EMI ?"

input2: "I want EMI for a loan of 200000 at 10 percent for 2 years ?"

input3_SPELL_ERROR: "What is the simpl interst on 5000 at 5 percent for 3 years ?" 

input4: "I am living in Moscow. The simple iterest is 5% there on loans. 
         I took a loan on simple interest for 3 yeas . The loan amt. was 5000. 
         What is the toal simple interest I would end up paying ?"

In [21]:
# 1) 
user_input = "I took a loan of 2 lakh at 10% for 2 years, what's my EMI ?"

response = agent.invoke(
    {"messages": [{"role": "user", 
                   "content": user_input
                  }]
    }
)

print("Response:", response)
print("Final Answer:", response["messages"][-1].content)

...Running EMI calculator tool
Response: {'messages': [HumanMessage(content="I took a loan of 2 lakh at 10% for 2 years, what's my EMI ?", additional_kwargs={}, response_metadata={}, id='653803a8-c2f1-4107-a78c-67ff6fdba662'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 110, 'total_tokens': 134, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_9490d6845c', 'id': 'chatcmpl-DZt5bjUghx8LILWXNWnj0q6btEIl4', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dd803-b07d-7f23-97e3-3e7e8091752d-0', tool_calls=[{'name': 'calculate_emi', 'args': {'principal': 200000, 'rate': 10, 'time': 2}, 'id': 'call_GjimbMe0hnuxRcRO

In [22]:
# 2) Structured output: Read that from the response

for msg in response["messages"]:
    if msg.type == "tool":
        print("Structured Output:", msg.content)

Structured Output: {"principal": 200000.0, "rate": 10.0, "time": 2.0, "emi": 9228.99}


In [23]:
# 1) 
user_input = "I want EMI for a loan of 200000 at 10 percent for 2 years ?"

response = agent.invoke(
    {"messages": [{"role": "user", 
                   "content": user_input
                  }]
    }
)

print("Response:", response)
print("Final Answer:", response["messages"][-1].content)

...Running EMI calculator tool
Response: {'messages': [HumanMessage(content='I want EMI for a loan of 200000 at 10 percent for 2 years ?', additional_kwargs={}, response_metadata={}, id='339fb5f6-16c6-48fd-9c7e-52bd9360d816'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 108, 'total_tokens': 132, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_9490d6845c', 'id': 'chatcmpl-DZt5tjyWZoTaQF8XbaW1Zm6RCT5W2', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dd803-fbdf-7c82-9df5-35c204892877-0', tool_calls=[{'name': 'calculate_emi', 'args': {'principal': 200000, 'rate': 10, 'time': 2}, 'id': 'call_mEz1WACVZHLVR1a2

In [24]:
# 2) Structured output: Read that from the response

for msg in response["messages"]:
    if msg.type == "tool":
        print("Structured Output:", msg.content)

Structured Output: {"principal": 200000.0, "rate": 10.0, "time": 2.0, "emi": 9228.99}


In [25]:
# 1) 
user_input = "What is the simpl interst on 5000 at 5 percent for 3 years ?"

response = agent.invoke(
    {"messages": [{"role": "user", 
                   "content": user_input
                  }]
    }
)

print("Response:", response)
print("Final Answer:", response["messages"][-1].content)

...Running simple interest tool
Response: {'messages': [HumanMessage(content='What is the simpl interst on 5000 at 5 percent for 3 years ?', additional_kwargs={}, response_metadata={}, id='0f55c736-1c3f-446c-88f6-2a4349c7bcdb'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 108, 'total_tokens': 132, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_9490d6845c', 'id': 'chatcmpl-DZt68XH7ANEHJWO9pF413L9pSLw3R', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dd804-33aa-7020-a0c6-ac87ce7478a5-0', tool_calls=[{'name': 'calculate_simple_interest', 'args': {'principal': 5000, 'rate': 5, 'time': 3}, 'id': 'call_Gcky2

In [26]:
# 2) Structured output: Read that from the response

for msg in response["messages"]:
    if msg.type == "tool":
        print("Structured Output:", msg.content)

Structured Output: {"principal": 5000.0, "rate": 5.0, "time": 3.0, "interest": 750.0}


In [27]:
# 1) 
user_input = "I am living in Moscow. The simple iterest is 5% there on loans.\
              I took a loan on simple interest for 3 yeas . The loan amt. was 5000 . \
              What is the toal simple interest I would end up paying ?"

response = agent.invoke(
    {"messages": [{"role": "user", 
                   "content": user_input
                  }]
    }
)

print("Response:", response)
print("Final Answer:", response["messages"][-1].content)

...Running simple interest tool
Response: {'messages': [HumanMessage(content='I am living in Moscow. The simple iterest is 5% there on loans.              I took a loan on simple interest for 3 yeas . The loan amt. was 5000 .               What is the toal simple interest I would end up paying ?', additional_kwargs={}, response_metadata={}, id='064b7e11-6b2c-4a82-b7b6-1c9e6fa3d1a9'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 143, 'total_tokens': 167, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_9490d6845c', 'id': 'chatcmpl-DZt6htinlLhtKpy1rluv97fwhmkhT', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--0

In [28]:
# 2) Structured output: Read that from the response

for msg in response["messages"]:
    if msg.type == "tool":
        print("Structured Output:", msg.content)

Structured Output: {"principal": 5000.0, "rate": 5.0, "time": 3.0, "interest": 750.0}
